# VisionBridge — trained model check + CTC overfit sanity (Colab)

**Inference/sanity-check only.** This notebook does not perform full training.

If the processed dataset is missing, it downloads the real sentence-level ISL-CSLTR videos with KaggleHub, extracts real MediaPipe Holistic features for a tiny valid subset, creates `data/processed/isltranslate/`, and runs the repository's CTC overfit sanity test.

Decision gate: **OVERFIT SANITY: PASS → full training may proceed. FAIL → diagnose first.**

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_ROOT = Path('/content/VisionBridge')
if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','main'],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)
print('Repo:',REPO_ROOT)
print('HEAD:',subprocess.check_output(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],text=True).strip())
print('BRANCH:',subprocess.check_output(['git','-C',str(REPO_ROOT),'branch','--show-current'],text=True).strip())
print('SYNC: PASS')


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
MP_ENV = Path('/content/visionbridge_mp312')
MP_PYTHON = MP_ENV / 'bin/python'
MPL_CONFIG = Path('/content/visionbridge_mplconfig')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
uv = shutil.which('uv')
if uv is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','uv'],check=True)
    uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv is not available on PATH; restart Colab and rerun.')
if subprocess.run([uv,'python','find','3.12'],capture_output=True).returncode != 0:
    subprocess.run([uv,'python','install','3.12'],check=True)
if not MP_PYTHON.exists():
    subprocess.run([uv,'venv','--python','3.12',str(MP_ENV)],check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(MPL_CONFIG)
probe = subprocess.run([str(MP_PYTHON),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'],text=True,capture_output=True,env=env)
if probe.returncode != 0 or probe.stdout.strip() != '0.10.21':
    subprocess.run([uv,'pip','install','--python',str(MP_PYTHON),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'],check=True,env=env)
probe = subprocess.run([str(MP_PYTHON),'-c','import sys,os,mediapipe; from mediapipe.python.solutions import holistic; print(sys.version.split()[0]); print(mediapipe.__version__); print(os.environ.get("MPLBACKEND"))'],text=True,capture_output=True,env=env)
print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError('Isolated MediaPipe environment failed validation.')


In [ ]:
import torch, shutil
from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model
WEIGHTS = REPO_ROOT / 'backend/app/models/weights/base_model.pt'
VOCAB = REPO_ROOT / 'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print('Checkpoint/vocab missing from repo. Upload both files now.')
    uploaded = files.upload()
    for name in ('base_model.pt','base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(name)
        WEIGHTS.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy(name,WEIGHTS.parent/name)
tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS,map_location='cpu',weights_only=True)
assert isinstance(state,dict) and 'output_head.weight' in state
assert int(state['output_head.weight'].shape[0]) == tokenizer.vocab_size
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_frozen_base_model(str(WEIGHTS),vocab_size=tokenizer.vocab_size).to(device).eval()
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 0
print('Device:',device)
print('Vocab:',tokenizer.vocab_size)
print('CHECKPOINT VALIDATION: PASS')


In [ ]:
import os, glob, subprocess, sys
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub'],check=True)
    import kagglehub
dataset_path = kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
roots = [p for p in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True) if os.path.isdir(p) and 'Video' in os.path.basename(p)]
if len(roots) != 1:
    raise RuntimeError(f'Expected one sentence-level video directory, found {len(roots)}: {roots}')
VIDEO_ROOT = roots[0]
video_files = []
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext),recursive=True))
video_files = sorted(video_files)
if not video_files:
    raise FileNotFoundError(f'No sentence-level videos found under {VIDEO_ROOT}')
print('Dataset:',dataset_path)
print('Video root:',VIDEO_ROOT)
print('Total real videos:',len(video_files))


In [ ]:
import numpy as np
from pathlib import Path
import subprocess, textwrap, csv
DATA_DIR = REPO_ROOT / 'data/processed/isltranslate'
CSV_PATH = DATA_DIR / 'ISLTranslate.csv'
POSE_DIR = DATA_DIR / 'pose'
FACE_DIR = DATA_DIR / 'face'
DATA_DIR.mkdir(parents=True,exist_ok=True)
POSE_DIR.mkdir(parents=True,exist_ok=True)
FACE_DIR.mkdir(parents=True,exist_ok=True)
existing_csv = CSV_PATH.exists()
existing_pose = list(POSE_DIR.glob('*.npy'))
existing_face = list(FACE_DIR.glob('*.npy'))
if existing_csv and existing_pose and existing_face:
    print('Processed dataset already exists. Reusing it.')
    print('CSV:',CSV_PATH)
    print('Pose files:',len(existing_pose))
    print('Face files:',len(existing_face))
else:
    helper = REPO_ROOT / 'data/model_check/_extract_for_training.py'
    helper.parent.mkdir(parents=True,exist_ok=True)
    helper.write_text(textwrap.dedent('''
import sys
from pathlib import Path
import numpy as np
from mediapipe.python.solutions import holistic
repo=Path(sys.argv[1]); video=sys.argv[2]; pose_out=Path(sys.argv[3]); face_out=Path(sys.argv[4])
sys.path.insert(0,str(repo/'backend'))
from scripts.extract_keypoints import extract_clip_keypoints
with holistic.Holistic(static_image_mode=False,model_complexity=1) as solution:
    pose,face=extract_clip_keypoints(video,solution)
assert pose.ndim==2 and pose.shape[1]==132
assert face.ndim==2 and face.shape[1]==1404
assert pose.shape[0]==face.shape[0] and pose.shape[0]>0
np.save(pose_out,pose); np.save(face_out,face)
print('frames=',pose.shape[0])
'''),encoding='utf-8')
    selected=[]
    print('Preparing a tiny REAL processed dataset for the overfit sanity test...')
    for video in video_files:
        uid=Path(video).stem
        text=Path(video).parent.name.replace('_',' ').strip()
        if not text:
            continue
        pose_path=POSE_DIR/f'{uid}.npy'
        face_path=FACE_DIR/f'{uid}.npy'
        result=subprocess.run([str(MP_PYTHON),str(helper),str(REPO_ROOT),video,str(pose_path),str(face_path)],text=True,capture_output=True,env=env)
        if result.returncode != 0:
            print('Skipping extraction failure:',video)
            print(result.stderr[-500:])
            continue
        pose=np.load(pose_path,mmap_mode='r')
        frame_count=int(pose.shape[0])
        if frame_count < len(text):
            pose_path.unlink(missing_ok=True); face_path.unlink(missing_ok=True)
            continue
        selected.append({'uid':uid,'text':text,'frames':frame_count,'video':video})
        print(f"Selected {len(selected)}: frames={frame_count}, label_len={len(text)}, text={text!r}")
        if len(selected) >= 4:
            break
    if len(selected) < 4:
        raise RuntimeError(f'Could only prepare {len(selected)} valid real examples; need at least 4.')
    valid_uids={row['uid'] for row in selected}
    for path in POSE_DIR.glob('*.npy'):
        if path.stem not in valid_uids:
            path.unlink()
    for path in FACE_DIR.glob('*.npy'):
        if path.stem not in valid_uids:
            path.unlink()
    with CSV_PATH.open('w',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=['uid','text'])
        writer.writeheader()
        for row in selected:
            writer.writerow({'uid':row['uid'],'text':row['text']})
    print('REAL PROCESSED DATASET READY')
    print('CSV:',CSV_PATH)
    print('Pose files:',len(list(POSE_DIR.glob('*.npy'))))
    print('Face files:',len(list(FACE_DIR.glob('*.npy'))))


In [ ]:
import csv
import numpy as np
print('Data directory:',DATA_DIR)
for p in (CSV_PATH,POSE_DIR,FACE_DIR):
    print(f'{p}: {"FOUND" if p.exists() else "MISSING"}')
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Missing metadata CSV: {CSV_PATH}')
if not POSE_DIR.exists() or not FACE_DIR.exists():
    raise FileNotFoundError('Missing pose/face processed directories.')
with CSV_PATH.open(encoding='utf-8') as handle:
    rows=list(csv.DictReader(handle))
if len(rows) < 4:
    raise RuntimeError(f'Need at least 4 processed examples, found {len(rows)}')
for row in rows[:4]:
    uid=row['uid']
    pose=np.load(POSE_DIR/f'{uid}.npy',mmap_mode='r')
    face=np.load(FACE_DIR/f'{uid}.npy',mmap_mode='r')
    assert pose.ndim==2 and pose.shape[1]==132, pose.shape
    assert face.ndim==2 and face.shape[1]==1404, face.shape
    assert pose.shape[0]==face.shape[0] > 0
    print(f"{uid}: pose={pose.shape}, face={face.shape}, text={row['text']!r}")
print('PROCESSED DATA CONTRACT: PASS')


In [ ]:
print("Running the repository's tiny REAL-DATA CTC overfit sanity test...")
print('This is NOT the full training run.\n')
cmd=[sys.executable,'-m','app.training.overfit_sanity','--data-dir',str(DATA_DIR),'--samples','4','--steps','150']
env2=os.environ.copy(); env2['PYTHONPATH']=str(BACKEND_ROOT)
result=subprocess.run(cmd,cwd=REPO_ROOT,env=env2,text=True,capture_output=True)
print(result.stdout)
if result.stderr: print('STDERR:\n',result.stderr)
if result.returncode != 0:
    print('DECISION: FAIL - DO NOT START FULL RETRAINING.')
    raise RuntimeError('OVERFIT SANITY FAILED. Diagnose the training/data/CTC pipeline before full training.')
print('DECISION: PASS - FULL RETRAINING MAY PROCEED.')


## Handoff

After the sanity test, report: PASS/FAIL, initial CTC loss, final CTC loss, loss reduction, final blank ratio, and decoded predictions. Only a PASS unlocks full retraining.